# Deep Q-Learning - Lunar Lander (Colab Version)

**Updated for Google Colab with Gymnasium**

Train an agent to land a lunar lander safely on the moon using Deep Q-Learning.

## Setup: Install Dependencies

In [ ]:
# Install required packages
!pip install -q gymnasium[box2d] imageio imageio-ffmpeg

## 1 - Import Packages

In [ ]:
import time
import random
import base64
from collections import deque, namedtuple

import gymnasium as gym
import numpy as np
import imageio
import tensorflow as tf
from IPython.display import HTML

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.losses import MSE
from tensorflow.keras.optimizers import Adam

# Set seeds
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

## 2 - Hyperparameters

In [ ]:
MEMORY_SIZE = 100_000
GAMMA = 0.995
ALPHA = 1e-3
NUM_STEPS_FOR_UPDATE = 4
MINIBATCH_SIZE = 64
TAU = 1e-3
E_DECAY = 0.995
E_MIN = 0.01

## 3 - Helper Functions

In [ ]:
def get_experiences(memory_buffer):
    experiences = random.sample(memory_buffer, k=MINIBATCH_SIZE)
    states = tf.convert_to_tensor(np.array([e.state for e in experiences if e is not None]), dtype=tf.float32)
    actions = tf.convert_to_tensor(np.array([e.action for e in experiences if e is not None]), dtype=tf.float32)
    rewards = tf.convert_to_tensor(np.array([e.reward for e in experiences if e is not None]), dtype=tf.float32)
    next_states = tf.convert_to_tensor(np.array([e.next_state for e in experiences if e is not None]), dtype=tf.float32)
    done_vals = tf.convert_to_tensor(np.array([e.done for e in experiences if e is not None]).astype(np.uint8), dtype=tf.float32)
    return (states, actions, rewards, next_states, done_vals)

def check_update_conditions(t, num_steps_upd, memory_buffer):
    return (t + 1) % num_steps_upd == 0 and len(memory_buffer) > MINIBATCH_SIZE

def get_new_eps(epsilon):
    return max(E_MIN, E_DECAY * epsilon)

def get_action(q_values, epsilon=0):
    if random.random() > epsilon:
        return np.argmax(q_values.numpy()[0])
    else:
        return random.choice(np.arange(4))

def update_target_network(q_network, target_q_network):
    for target_weights, q_net_weights in zip(target_q_network.weights, q_network.weights):
        target_weights.assign(TAU * q_net_weights + (1.0 - TAU) * target_weights)

def embed_mp4(filename):
    video = open(filename, 'rb').read()
    b64 = base64.b64encode(video).decode()
    return HTML(f'<video width="840" height="480" controls><source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>')

def create_video(filename, env, q_network, fps=30):
    with imageio.get_writer(filename, fps=fps) as video:
        done = False
        state, _ = env.reset()
        frame = env.render()
        video.append_data(frame)
        while not done:
            state = np.expand_dims(state, axis=0)
            q_values = q_network(state)
            action = np.argmax(q_values.numpy()[0])
            state, _, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            frame = env.render()
            video.append_data(frame)

## 4 - Load the Environment

In [ ]:
env = gym.make('LunarLander-v2', render_mode='rgb_array')
state_size = env.observation_space.shape
num_actions = env.action_space.n

print('State Shape:', state_size)
print('Number of actions:', num_actions)

## 5 - Explore the Environment

In [ ]:
initial_state, _ = env.reset()
action = 0
next_state, reward, terminated, truncated, info = env.step(action)

print("Initial State:", initial_state)
print("Action:", action)
print("Next State:", next_state)
print("Reward:", reward)
print("Terminated:", terminated)

## 6 - Exercise 1: Create Q-Networks

Create the Q-Network and Target Q-Network with:
- Input layer: state_size
- Hidden layer 1: 64 units, ReLU
- Hidden layer 2: 64 units, ReLU
- Output layer: num_actions, linear

In [ ]:
# EXERCISE 1: Create Q-Networks

q_network = Sequential([
    Input(shape=state_size),
    Dense(units=64, activation='relu'),
    Dense(units=64, activation='relu'),
    Dense(units=num_actions, activation='linear'),
])

target_q_network = Sequential([
    Input(shape=state_size),
    Dense(units=64, activation='relu'),
    Dense(units=64, activation='relu'),
    Dense(units=num_actions, activation='linear'),
])

optimizer = Adam(learning_rate=ALPHA)

## 7 - Exercise 2: Compute Loss Function

In [ ]:
# EXERCISE 2: Compute Loss

def compute_loss(experiences, gamma, q_network, target_q_network):
    states, actions, rewards, next_states, done_vals = experiences
    max_qsa = tf.reduce_max(target_q_network(next_states), axis=-1)
    
    # Set y = R if episode terminates, otherwise y = R + γ max Q^(s,a)
    y_targets = rewards + (gamma * max_qsa * (1 - done_vals))
    
    q_values = q_network(states)
    q_values = tf.gather_nd(q_values, tf.stack([tf.range(q_values.shape[0]), tf.cast(actions, tf.int32)], axis=1))
    
    loss = MSE(y_targets, q_values)
    return loss

## 8 - Agent Learning Function

In [ ]:
@tf.function
def agent_learn(experiences, gamma):
    with tf.GradientTape() as tape:
        loss = compute_loss(experiences, gamma, q_network, target_q_network)
    gradients = tape.gradient(loss, q_network.trainable_variables)
    optimizer.apply_gradients(zip(gradients, q_network.trainable_variables))
    update_target_network(q_network, target_q_network)

## 9 - Train the Agent

In [ ]:
start = time.time()

num_episodes = 2000
max_num_timesteps = 1000
total_point_history = []
num_p_av = 100
epsilon = 1.0

experience = namedtuple("Experience", field_names=["state", "action", "reward", "next_state", "done"])
memory_buffer = deque(maxlen=MEMORY_SIZE)
target_q_network.set_weights(q_network.get_weights())

for i in range(num_episodes):
    state, _ = env.reset()
    total_points = 0
    
    for t in range(max_num_timesteps):
        state_qn = np.expand_dims(state, axis=0)
        q_values = q_network(state_qn)
        action = get_action(q_values, epsilon)
        
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        
        memory_buffer.append(experience(state, action, reward, next_state, done))
        
        update = check_update_conditions(t, NUM_STEPS_FOR_UPDATE, memory_buffer)
        if update:
            experiences = get_experiences(memory_buffer)
            agent_learn(experiences, GAMMA)
        
        state = next_state.copy()
        total_points += reward
        
        if done:
            break
    
    total_point_history.append(total_points)
    av_latest_points = np.mean(total_point_history[-num_p_av:])
    epsilon = get_new_eps(epsilon)
    
    print(f"\rEpisode {i+1} | Total point average of the last {num_p_av} episodes: {av_latest_points:.2f}", end="")
    
    if (i+1) % num_p_av == 0:
        print(f"\rEpisode {i+1} | Total point average of the last {num_p_av} episodes: {av_latest_points:.2f}")
    
    if av_latest_points >= 200.0:
        print(f"\n\nEnvironment solved in {i+1} episodes!")
        q_network.save('lunar_lander_model.h5')
        break

tot_time = time.time() - start
print(f"\nTotal Runtime: {tot_time:.2f} s ({(tot_time/60):.2f} min)")

## 10 - Plot Training History

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

plt.figure(figsize=(10, 7))
plt.plot(total_point_history, linewidth=1, alpha=0.4)
plt.plot(pd.Series(total_point_history).rolling(20).mean(), linewidth=2)
plt.xlabel('Episode')
plt.ylabel('Total Points')
plt.title('Training Progress')
plt.grid(True)
plt.show()

## 11 - See the Trained Agent

In [ ]:
filename = "lunar_lander.mp4"
create_video(filename, env, q_network)
embed_mp4(filename)